# 5. Modelli predittivi di machine learning

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FedeGambe/Master_s_thesis_Data_science/blob/main/notebooks/05_machine_learning.ipynb)

Confronto di 15 algoritmi e ottimizzazione dei migliori, senza data leakage. I risultati sono prodotti da `scripts/05_machine_learning.py` (circa 20 minuti) e salvati in `risultati/`: il notebook li carica. Impostare `RIESEGUI = True` per ricalcolare tutto.

In [ ]:
# Configurazione dell'ambiente (locale o Google Colab)
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("Master_s_thesis_Data_science"):
        !git clone -q -b main https://github.com/FedeGambe/Master_s_thesis_Data_science.git
    %cd Master_s_thesis_Data_science
    !pip install -q -r requirements-colab.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import plotly.io as pio
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)
if os.environ.get("TESI_GRAFICI_STATICI"):
    pio.renderers.default = "png"  # usato solo per salvare output visibili su GitHub
    pio.defaults.default_width = 1000

In [ ]:
RIESEGUI = False

from pathlib import Path
import numpy as np
from tesi_bev import archivio as A, config as C, dati, grafici as G
if RIESEGUI or not Path("risultati/machine_learning/risultati_test.csv").exists():
    !python scripts/05_machine_learning.py
SEZ = "machine_learning"

## Cosa è cambiato rispetto alla tesi

| Problema nella versione originale | Correzione |
|---|---|
| `scaler.fit_transform(X_test)`: scaler riaddestrato sul test | Scaler dentro una `Pipeline`, addestrato solo sui fold di training |
| Selezione delle feature su tutto il dataset | `SelectKBest` dentro la pipeline, `k` ottimizzato in CV |
| Stimatore del "fold migliore" usato sul test | `refit=True`: modello riaddestrato su tutto il training |
| Split diversi tra modelli (`random_state` 343 e 42) | Un unico split stratificato per tutti |
| Tabella finale con metriche su righe sbagliate | Metriche calcolate da un'unica funzione per modello |
| Solo accuracy | ROC-AUC per l'ottimizzazione, IC bootstrap, test di McNemar, calibrazione |

In [ ]:
A.carica_json(SEZ, 'split')

## 5.1 Screening in cross-validation (solo training)

In [ ]:
screening = A.carica_tabella(SEZ, "screening_cv", indice=False)
G.box_screening(screening, "roc_auc").show()
A.carica_tabella(SEZ, "screening_riepilogo")

## 5.2 Iperparametri ottimizzati

In [ ]:
ottimizzazione = A.carica_json(SEZ, "ottimizzazione")
pd.DataFrame({k: {"ROC-AUC CV": v.get("roc_auc_cv"), "tempo (s)": v.get("tempo_s"), "parametri": v.get("parametri", v.get("componenti"))}
              for k, v in ottimizzazione.items()}).T

## 5.3 Risultati sul test set

In [ ]:
risultati = A.carica_tabella(SEZ, "risultati_test")
G.barre_metriche(risultati, ic=risultati).show()
risultati[["accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "roc_auc_ic_basso", "roc_auc_ic_alto", "brier"]]

**Il migliore è davvero migliore?** Il test di McNemar confronta gli errori del modello migliore con quelli di ogni
altro modello sullo stesso test set. Un p-value alto significa che la differenza non è statisticamente significativa.

In [ ]:
A.carica_tabella(SEZ, 'mcnemar_vs_migliore')

In [ ]:
roc = A.carica_json(SEZ, "curve_roc")
G.curve_roc({k: v for k, v in roc.items() if k in risultati.index[:5].tolist() + ["Regressione logistica"]}).show()
cal = A.carica_json(SEZ, "calibrazione")
G.calibrazione({k: cal[k] for k in risultati.index[:3]}).show()

In [ ]:
migliore = risultati.index[0]
G.matrice_confusione(risultati.loc[migliore], f"Matrice di confusione - {migliore}").show()

## 5.4 Confronto con i risultati della tesi

In [ ]:
A.carica_tabella(SEZ, 'confronto_tesi_revisione')

## 5.5 Quanto pesa l'auto precedente?

*Auto precedente: BEV* è il predittore più forte, ma anche il meno informativo dal punto di vista di marketing: chi
aveva già un BEV tende a ricomprarlo. Togliendolo si misura quanto le altre caratteristiche spiegano da sole.

In [ ]:
A.carica_tabella(SEZ, 'ablation_bev_precedente')[['accuracy', 'f1', 'roc_auc']]

## 5.6 Interpretabilità: permutation importance e SHAP

In [ ]:
G.importanza(A.carica_tabella(SEZ, "importanza_permutazione", indice=False), "importanza",
             "Permutation importance (calo di ROC-AUC sul test)").show()
G.importanza(A.carica_tabella(SEZ, "importanza_shap", indice=False), "shap_medio_assoluto",
             "Importanza SHAP (|valore| medio)").show()

In [ ]:
import shap, matplotlib.pyplot as plt
valori = np.load("risultati/machine_learning/shap_valori.npy")
campione = pd.read_csv("risultati/machine_learning/shap_campione.csv")
shap.summary_plot(valori, campione.rename(columns=C.nome_breve), show=False)
plt.title(f"SHAP - {A.carica_json(SEZ, 'modello_finale')['modello_shap']}")
plt.show()